# ML-10 — Action Playbook

Translate model output into ranked editorial recommendations. This is the 'so what' — what a content team does with the model's scores.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Setup — load the ranked queue

In [ ]:
import pandas as pd
import numpy as np

# The pipeline generates this queue — load it if available, else use the sample
try:
    queue = pd.read_csv("../../outputs/refresh_queue.csv")
except FileNotFoundError:
    queue = pd.read_csv("../../outputs/refresh_queue_sample.csv")

print(f"Queue: {len(queue):,} pages ranked")
print(f"Columns: {list(queue.columns[:10])}...")
print(f"\nAction distribution:")
print(queue["suggested_action"].value_counts().to_string())

## 2. The five action tiers

The queue assigns every page one of five actions, ordered by urgency:

### Tier 1: Refresh & Review CTR (highest priority)
**6,657 pages.** High impressions + low CTR (<0.5%). These pages are *visible but not compelling* — titles, meta descriptions, and opening content likely need refreshing. The traffic exists; the page just isn't capturing clicks.

### Tier 2: Refresh
**8,178 pages.** General content refresh — declining with demand, model-flagged decline risk, or stale visibility. Update content, add new information, review internal linking, re-optimize for current search intent.

### Tier 3: Refresh & Review Engagement
**1,990 pages.** Sufficient sessions (≥30) but low engagement rate (<30%) or low scroll rate (<30%). Traffic arrives, but readers leave quickly. Focus on content quality, page structure, and reader experience.

### Tier 4: Expand & Refresh
**82 pages.** Thin pages (<1,200 words) with meaningful visibility (≥250 impressions). Content-depth gaps — expanding with substantive content may unlock latent ranking potential.

### Tier 5: Monitor
**13,093 pages.** No urgent action signals. Continue monitoring — re-score monthly to catch new declines early.

## 3. Confidence tiers

In [ ]:
print("Confidence distribution:")
print(queue["confidence"].value_counts().to_string())

print(f"\nHigh-confidence criteria:")
print(f"  - Final score ≥ 80th percentile")
print(f"  - Impressions ≥ 500")
print(f"  - Sessions ≥ 10")
print(f"  - Model probability ≥ 0.5")
print(f"  → Only pages meeting ALL four criteria are 'high confidence'")
print(f"")
print(f"Recommendation: Start with high-confidence pages. These have the")
print(f"strongest signal AND enough traffic to make a refresh worthwhile.")

## 4. Top-10 queue preview

In [ ]:
preview_cols = ["final_rank", "final_refresh_score", "best_model_probability",
                "suggested_action", "confidence", "impressions_90d", "sessions_90d",
                "avg_position", "trend_direction"]

available_cols = [c for c in preview_cols if c in queue.columns]
top10 = queue.head(10)[available_cols]

print("Top 10 pages in the refresh queue:")
print(top10.to_string(index=False))

## 5. Reason codes — the 'why' behind every recommendation

In [ ]:
# Count reason codes across the entire queue
reason_col = "final_reason_codes" if "final_reason_codes" in queue.columns else "reason_codes"
if reason_col in queue.columns:
    reason_counts = {}
    for codes in queue[reason_col].dropna():
        for code in str(codes).split("|"):
            reason_counts[code] = reason_counts.get(code, 0) + 1
    
    print("Reason code frequency (across all 30,000 pages):")
    for reason, count in sorted(reason_counts.items(), key=lambda x: -x[1]):
        pct = count / len(queue) * 100
        print(f"  {reason:35s}  {count:6,}  ({pct:5.1f}%)")
    
    print(f"\nEvery page gets reason codes — not just a score, but 'why.'")
    print(f"This makes every recommendation explainable and auditable.")
else:
    print("Reason codes column not found — run the full pipeline first.")

## 6. How to use this playbook

### For the content strategist:
1. **Start at Tier 1** (Refresh & Review CTR) — these are your quick wins. High visibility, low clicks = metadata problem.
2. **Then Tier 2** (Refresh) — bulk content refresh for declining pages with demand.
3. **Tier 3** (Engagement) — deeper content-quality issues, harder to fix but high impact.
4. **Tier 4** (Expand) — rare but clear gaps.
5. **Tier 5** (Monitor) — re-score monthly, don't ignore permanently.

### Critical caveats:
- **This is a reviewer aid, not an auto-publisher.** Always verify the page manually before acting.
- **The reason codes tell you WHY.** Use them to tailor the refresh approach.
- **High confidence ≠ certainty.** Even high-confidence pages should be editorially reviewed.
- **Re-score monthly.** Today's 'monitor' page may be tomorrow's 'refresh' candidate.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.